In [25]:
import numpy as np
import pandas as pd
import sys,os

from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.pardir)

df = pd.read_csv('../Datas/ratings.csv')
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [26]:
df.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')

In [27]:
ratings = df.pivot(index="userId",columns='movieId',values='rating')
ratings_ = ratings.fillna(0)

In [28]:
from sklearn.metrics.pairwise import cosine_similarity
user_sim_df = cosine_similarity(ratings_)
user_sim_df= pd.DataFrame(user_sim_df,index=ratings.index, columns=ratings.index)


In [36]:
def recommend(user, ratings, user_sim_df, top_n=5):
    # 해당 사용자가 안 본 영화 목록
    unseen = ratings.loc[user][ratings.loc[user].isna()].index

    scores = {}
    for movie in unseen:
        # 이 영화를 본 다른 사용자들
        watched = ratings[movie].dropna().index
        watched = watched[watched != user]

        if len(watched) == 0:
            continue

        # 유사도 가중 평균: sum(유사도 * 평점) / sum(유사도)
        sim = user_sim_df.loc[user, watched]
        rate = ratings.loc[watched, movie]
        scores[movie] = np.dot(sim, rate) / sim.sum()

    result = pd.Series(scores).sort_values(ascending=False)
    return result.head(top_n)

In [37]:
# 유저B에게 추천
print('유저2가 안 본 영화:', sum(list(ratings.loc[2][ratings.loc[2].isna()].index)))
print()
print('2한테 추천 결과 (예상 평점):')
recommend(2, ratings, user_sim_df)

유저B가 안 본 영화: 408750459

추천 결과 (예상 평점):


C:\Users\hyuno\AppData\Local\Temp\ipykernel_16696\3330905512.py:17: RuntimeWarning: invalid value encountered in scalar divide
  scores[movie] = np.dot(sim, rate) / sim.sum()


82744     5.0
148       5.0
172875    5.0
108795    5.0
102084    5.0
dtype: float64